# 📊 W4: F&B 채널별 매출 분석
**hexa-2 — Week 4** | ⏱️ 60분 | 🎯 배달앱별 매출 비교 + Gemini 전략

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
# === 2026년 3월 기준 최신 모델 선택 ===
# ✅ GA(안정): gemini-2.5-flash  ← 기본값 (권장)
# 🔵 Preview : gemini-3.1-flash-lite-preview  (최신, 2026.3 출시)
# 🔵 Preview : gemini-3.1-pro-preview  (최강, 복잡한 분석용)
# ⚠️  구버전 gemini-2.0-flash 는 2026년 6월 1일 종료 예정
MODEL_NAME = 'gemini-2.5-flash'  # 원하는 모델로 변경하세요
model=genai.GenerativeModel('gemini-2.5-flash'); print('✅ 준비 완료')


## Step 1: 가게 정보 입력 (✏️)

In [ ]:
INFO={'name':'✏️ [가게명]','channels':['배달의민족','쿠팡이츠','자사배달']}
print('✅',INFO['name'])


## Step 2: 매출 데이터 로드

In [ ]:
import io, requests, pandas as pd
URL='https://raw.githubusercontent.com/Reasonofmoon/hexa-2/main/shared/fnb_sales_sample.csv'
try:
    df=pd.read_csv(URL,encoding='utf-8-sig'); print(f'✅ 로드: {len(df)}행')
except:
    try:
        from google.colab import files; up=files.upload()
        if up: df=pd.read_csv(io.BytesIO(list(up.values())[0]),encoding='utf-8-sig')
        else: raise Exception('취소')
    except:
        import numpy as np; np.random.seed(42)
        df=pd.DataFrame({'날짜':['2026-01-'+str(d).zfill(2) for d in range(1,31)],
            '메뉴명':['김치찌개','된장찌개','순두부']*10,'주문수':np.random.randint(30,150,30),'매출':np.random.randint(300000,1200000,30)})
        print('📋 기본 샘플 사용')
print(df.head())


## Step 3: 채널별 매출 비교 시각화

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig,axes=plt.subplots(1,2,figsize=(12,4))
if '메뉴명' in df.columns and '매출' in df.columns:
    menu_sales=df.groupby('메뉴명')['매출'].sum()
    menu_sales.plot(kind='bar',ax=axes[0],color='steelblue'); axes[0].set_title('메뉴별 총 매출')
    df['날짜']=pd.to_datetime(df['날짜'])
    daily=df.groupby('날짜')['매출'].sum()
    daily.plot(ax=axes[1],color='coral'); axes[1].set_title('일별 매출 추세')
else:
    axes[0].text(0.5,0.5,'데이터 없음',ha='center'); axes[1].text(0.5,0.5,'데이터 없음',ha='center')
plt.tight_layout(); plt.savefig('sales_chart.png',dpi=150,bbox_inches='tight'); plt.show()


## Step 4: Gemini 채널 전략 분석

In [ ]:
total=df['매출'].sum() if '매출' in df.columns else 0
p=f"""F&B 매출 분석 전문가로서 채널 전략을 제안하세요.
가게: {INFO['name']}, 총 매출: {total:,.0f}원
채널: {INFO['channels']}
채널 최적화 전략 3가지 + 즉시 실행 방안. 마크다운."""
resp=model.generate_content(p)
print(resp.text)


## Step 5: 다운로드

In [ ]:
with open('sales_report.md','w',encoding='utf-8') as f: f.write(resp.text)
from google.colab import files
files.download('sales_report.md'); files.download('sales_chart.png')
print('✅ 완료!')
